## 데이터 통합 및 어뷰징 분석

In [1]:
import pandas as pd
import numpy as np

## STEP 1. 데이터 로드 및 기본 전처리

In [2]:
df_master    = pd.read_csv('/Users/joshuakim/Desktop/스파르타 최종데이터/광고 목록_도메인 라벨링_결측치처리.csv', low_memory=False)
df_reward    = pd.read_csv('/Users/joshuakim/Desktop/스파르타 최종데이터/IVE_광고_데이터_스파르타/IVE_광고적립_all.csv', low_memory=False)
df_engagement= pd.read_csv('/Users/joshuakim/Desktop/스파르타 최종데이터/IVE_광고참여정보_all.csv', low_memory=False)
df_report    = pd.read_csv('/Users/joshuakim/Desktop/스파르타 최종데이터/아이브시간대별광고리포트_1년치_all.csv', low_memory=False)

In [3]:
# df_master: 날짜형 변환 + edate 상한 보정
for col in ['regdate', 'ads_sdate', 'ads_edate']:
    df_master[col] = pd.to_datetime(df_master[col], errors='coerce')
df_master['ads_edate'] = df_master['ads_edate'].clip(upper=pd.to_datetime('2025-08-29'))

# df_master2: 비용 이상치 제거 전 원본 (CTIT 계산·조인용)
df_master2 = df_master.copy()

type_map = {
    1:'설치형', 2:'실행형', 3:'참여형', 4:'클릭형',
    5:'페이스북', 6:'트위터', 7:'인스타그램', 8:'노출형',
    9:'퀘스트', 10:'유튜브', 11:'네이버', 12:'CPS(구매)'
}
domain_map = {1:'엔터테인먼트', 2:'금융', 3:'라이프스타일', 4:'커머스', 5:'기타'}

df_master2['ads_type_name'] = df_master2['ads_type'].map(type_map)
df_master2['domain_name']   = df_master2['domain_label'].map(domain_map)

df_reward['advid'] = df_reward['advid'].fillna('Unknown')

df_engagement = (
    df_engagement
    .drop(columns=['carrier'])
    .assign(network=lambda x: x['network'].fillna('Unknown'))
    .dropna(subset=['user_ip'])
    .reset_index(drop=True)
)

/var/folders/2m/_bqynwn904ndlfmk8kdlnn6w0000gn/T/ipykernel_92309/4277861064.py:3: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df_master[col] = pd.to_datetime(df_master[col], errors='coerce')


## STEP 2. 비용 구조 이상치 제거

비용 흐름 원칙: S(광고주) ≥ A(IVE) ≥ Earn(매체) ≥ RWD(유저), 모두 양수.  
이를 위반하거나 비용=0인 행은 데이터 오류로 제거.

In [4]:
df_master = df_master[
    (df_master['ads_contract_price'] >= df_master['ads_reward_price']) &
    (df_master['ads_contract_price'] > 0) &
    (df_master['ads_reward_price']   > 0)
].reset_index(drop=True)

df_engagement = df_engagement[
    (df_engagement['adv_price']      >= df_engagement['contract_price']) &
    (df_engagement['contract_price'] >= df_engagement['media_price']) &
    (df_engagement['media_price']    >= df_engagement['reward_price']) &
    (df_engagement[['adv_price', 'contract_price', 'media_price']].min(axis=1) > 0)
].reset_index(drop=True)

df_reward = df_reward[
    (df_reward['show_cost'] >= df_reward['adv_cost']) &
    (df_reward['adv_cost']  >= df_reward['earn_cost']) &
    (df_reward['earn_cost'] >= df_reward['rwd_cost']) &
    (df_reward[['show_cost', 'adv_cost', 'earn_cost']].min(axis=1) > 0)
].reset_index(drop=True)

# reward에만 있고 engagement에 없는 이상 케이스 제거
valid_click_keys = set(df_engagement['click_key'])
before = len(df_reward)
df_reward = df_reward[df_reward['click_key'].isin(valid_click_keys)].reset_index(drop=True)
print(f"reward 이상케이스 제거: {before:,} → {len(df_reward):,} (제거 {before - len(df_reward)}건)")

reward 이상케이스 제거: 1,470,614 → 1,470,533 (제거 81건)


## STEP 3. 파생변수 생성

**CTIT 이상치 라벨링**: 광고유형별 중앙값 × 0.1을 하한으로 설정. 클릭형(type=4)은 CTIT=0이면 오류.

**IP 통계**: 동일 IP에서 비정상적으로 많은 기기·클릭 수 = 어뷰징 의심.

In [5]:
# CTIT 이상치 라벨링
ctit_tmp = df_reward[['ads_idx', 'ctit']].merge(
    df_master2[['ads_idx', 'ads_type']], on='ads_idx', how='left'
)
threshold_map = (
    ctit_tmp[ctit_tmp['ads_type'] != 4]
    .groupby('ads_type')['ctit'].median() * 0.1
).to_dict()

ctit_tmp['threshold'] = ctit_tmp['ads_type'].map(threshold_map)
is_ctit_error = (
    ((ctit_tmp['ads_type'] == 4) & (ctit_tmp['ctit'] == 0)) |
    ((ctit_tmp['ads_type'] != 4) & (ctit_tmp['ctit'] < ctit_tmp['threshold']))
)
df_reward['ctit_error'] = np.where(is_ctit_error.values, 'Y', 'N')

# IP별 기기 수 / 클릭 수 집계
ip_stats = (
    df_engagement
    .groupby(['mda_idx', 'user_ip'])
    .agg(
        device_count=('dvc_idx',   'nunique'),
        click_count =('click_key', 'count')
    )
    .reset_index()
)

## STEP 4. 통합 DataFrame 생성

engagement(클릭) + reward(적립) + master(광고정보) + ip_stats(IP통계) 조인.  
`is_rewarded`: ctit가 존재하면 전환 완료(적립 발생) = 비용 청구 대상.

In [6]:
df_integrated = (
    df_engagement
    .merge(df_reward,    on='click_key',             how='left', suffixes=('', '_reward'))
    .merge(df_master2,   on='ads_idx',               how='left', suffixes=('', '_master'))
    .merge(ip_stats,     on=['mda_idx', 'user_ip'],  how='left')
)
df_integrated['is_rewarded'] = df_integrated['ctit'].notna().astype(int)

## STEP 5. 행 단위 어뷰징 분류 (row_label)

**등급 기준 (분위수)**
| 등급 | 기준 |
|------|------|
| 극단값 | 상위 0.1% 초과 → 확정 어뷰징 |
| 위험   | 상위 1.0% 초과 |
| 경고   | 상위 5.0% 초과 |
| 정상   | 상위 5.0% 이하 |

**가중치**: CTIT 50% / IP기기수 30% / IP클릭수 20%

**row_label**
- `미적립`: is_rewarded == 0
- `극단값`: 3개 지표 중 하나라도 극단값
- `어뷰징`: 극단값 아닌데 경고/위험 지표 존재
- `정상`: 모든 지표 정상

In [7]:
# 분위수 기준값 산출
d_crit = ip_stats['device_count'].quantile(0.999)
d_dang = ip_stats['device_count'].quantile(0.99)
d_warn = max(2, ip_stats['device_count'].quantile(0.95))

c_crit = ip_stats['click_count'].quantile(0.999)
c_dang = ip_stats['click_count'].quantile(0.99)
c_warn = ip_stats['click_count'].quantile(0.95)

print(f"[기기 수 기준]  정상: ≤{d_warn:.0f}  경고: ~{d_dang:.0f}  위험: ~{d_crit:.0f}  극단값: >{d_crit:.0f}")
print(f"[클릭 수 기준]  정상: ≤{c_warn:.0f}  경고: ~{c_dang:.0f}  위험: ~{c_crit:.0f}  극단값: >{c_crit:.0f}")

# ip_stats에 등급 부여
ip_stats['device_grade'] = np.select(
    [ip_stats['device_count'] <= d_warn,
     ip_stats['device_count'] <= d_dang,
     ip_stats['device_count'] <= d_crit],
    ['정상', '경고', '위험'], default='극단값'
)
ip_stats['click_grade'] = np.select(
    [ip_stats['click_count'] <= c_warn,
     ip_stats['click_count'] <= c_dang,
     ip_stats['click_count'] <= c_crit],
    ['정상', '경고', '위험'], default='극단값'
)
print(f"\n[device_grade 분포]\n{ip_stats['device_grade'].value_counts()}")
print(f"\n[click_grade 분포]\n{ip_stats['click_grade'].value_counts()}")

[기기 수 기준]  정상: ≤3  경고: ~6  위험: ~88  극단값: >88
[클릭 수 기준]  정상: ≤37  경고: ~258  위험: ~934  극단값: >934

[device_grade 분포]
device_grade
정상     928378
경고      24852
위험       7987
극단값       957
Name: count, dtype: int64

[click_grade 분포]
click_grade
정상     914222
경고      38340
위험       8649
극단값       963
Name: count, dtype: int64


In [8]:
# df_integrated에 IP 등급 조인
df_integrated = df_integrated.merge(
    ip_stats[['mda_idx', 'user_ip', 'device_grade', 'click_grade']],
    on=['mda_idx', 'user_ip'], how='left'
)
df_integrated['device_grade'] = df_integrated['device_grade'].fillna('정상')
df_integrated['click_grade']  = df_integrated['click_grade'].fillna('정상')

# CTIT 등급 부여
df_integrated['ctit_grade'] = np.select(
    [df_integrated['ctit_error'].isna(),
     df_integrated['ctit_error'] == 'Y',
     df_integrated['ctit_error'] == 'N'],
    ['미적립', '극단값', '정상'], default='미적립'
)
print(f"[ctit_grade 분포]\n{df_integrated['ctit_grade'].value_counts()}")

# 등급 → 점수 변환
grade_score_map = {'정상': 100, '미적립': 100, '경고': 70, '위험': 40, '극단값': 10}
df_integrated['ctit_score']   = df_integrated['ctit_grade'].map(grade_score_map)
df_integrated['device_score'] = df_integrated['device_grade'].map(grade_score_map)
df_integrated['click_score']  = df_integrated['click_grade'].map(grade_score_map)

# 가중 점수 및 row_label 분류
df_integrated['row_score'] = (
    df_integrated['ctit_score']   * 0.50 +
    df_integrated['device_score'] * 0.30 +
    df_integrated['click_score']  * 0.20
)

is_ctit_extreme   = df_integrated['ctit_grade']   == '극단값'
is_device_extreme = df_integrated['device_grade'] == '극단값'
is_click_extreme  = df_integrated['click_grade']  == '극단값'
is_device_bad     = df_integrated['device_grade'].isin(['경고', '위험', '극단값'])
is_click_bad      = df_integrated['click_grade'].isin(['경고', '위험', '극단값'])
is_extreme        = is_ctit_extreme | is_device_extreme | is_click_extreme
is_abusing        = (~is_extreme) & (is_device_bad | is_click_bad)
is_unrewarded     = df_integrated['is_rewarded'] == 0

df_integrated['row_label'] = np.select(
    [is_extreme, is_abusing, is_unrewarded],
    ['극단값', '어뷰징', '미적립'], default='정상'
)
print(f"\n[row_label 분포]\n{df_integrated['row_label'].value_counts()}")

[ctit_grade 분포]
ctit_grade
미적립    15360521
정상      1403981
극단값       66552
Name: count, dtype: int64

[row_label 분포]
row_label
어뷰징    7782515
극단값    6387902
미적립    1849072
정상      811565
Name: count, dtype: int64


## STEP 6. 매체 단위 위험도 스코어링

**[분류 방법 1] 점수 기반 (Risk_Label_score)** — 참고용  
Total_Score = ctit_score평균×0.50 + device_score평균×0.30 + click_score평균×0.20  
매우위험 ≤10 / 위험 <50 / 경고 <80 / 정상 ≥80

**[분류 방법 2] 비율 기반 (Risk_Label)** — 주력  
매우위험: extreme_ratio ≥ 30% / 위험: ≥10% 또는 abusing_ratio ≥50% / 경고: ≥3% 또는 abusing_ratio ≥20%

In [9]:
mda_score = (
    df_integrated
    .groupby('mda_idx')
    .agg(
        avg_ctit_score   = ('ctit_score',   'mean'),
        avg_device_score = ('device_score', 'mean'),
        avg_click_score  = ('click_score',  'mean'),
        extreme_ratio    = ('row_label', lambda x: (x == '극단값').mean()),
        abusing_ratio    = ('row_label', lambda x: (x == '어뷰징').mean()),
        normal_ratio     = ('row_label', lambda x: (x == '정상').mean()),
        unrewarded_ratio = ('row_label', lambda x: (x == '미적립').mean()),
        total_clicks     = ('click_key', 'count')
    )
    .reset_index()
)

mda_score['Total_Score'] = (
    mda_score['avg_ctit_score']   * 0.50 +
    mda_score['avg_device_score'] * 0.30 +
    mda_score['avg_click_score']  * 0.20
)
mda_score['Risk_Label_score'] = np.select(
    [mda_score['Total_Score'] <= 10,
     mda_score['Total_Score'] < 50,
     mda_score['Total_Score'] < 80,
     mda_score['Total_Score'] >= 80],
    ['매우위험', '위험', '경고', '정상'], default='위험'
)
mda_score['Risk_Label'] = np.select(
    [mda_score['extreme_ratio'] >= 0.30,
     (mda_score['extreme_ratio'] >= 0.10) | (mda_score['abusing_ratio'] >= 0.50),
     (mda_score['extreme_ratio'] >= 0.03) | (mda_score['abusing_ratio'] >= 0.20)],
    ['매우위험', '위험', '경고'], default='정상'
)

print(f"[Risk_Label 분포 (비율기반)]\n{mda_score['Risk_Label'].value_counts()}")
print(f"\n[Risk_Label_score 분포 (점수기반)]\n{mda_score['Risk_Label_score'].value_counts()}")
print(f"\n[하위 10개 매체 - 가장 위험]")
print(mda_score[['mda_idx','Total_Score','extreme_ratio','abusing_ratio','total_clicks','Risk_Label_score','Risk_Label']]
      .sort_values('Total_Score').head(10).to_string(index=False))

[Risk_Label 분포 (비율기반)]
Risk_Label
정상      135
경고       35
위험       12
매우위험      7
Name: count, dtype: int64

[Risk_Label_score 분포 (점수기반)]
Risk_Label_score
정상    181
경고      8
Name: count, dtype: int64

[하위 10개 매체 - 가장 위험]
 mda_idx  Total_Score  extreme_ratio  abusing_ratio  total_clicks Risk_Label_score Risk_Label
     562    53.865247       1.000000       0.000000         21454               경고       매우위험
     563    54.415948       1.000000       0.000000        540337               경고       매우위험
     294    54.849246       1.000000       0.000000          3582               경고       매우위험
     761    54.887896       1.000000       0.000000         71853               경고       매우위험
      58    59.844892       0.850827       0.143865        482794               경고       매우위험
      56    77.512897       0.266504       0.653799         40127               경고         위험
     539    79.464179       0.391515       0.549281      13468019               경고       매우위험
     482    79.799669     

## STEP 7. df_integrated에 매체 위험도 조인

In [10]:
cols_to_drop = ['Total_Score','Risk_Label_score','Risk_Label',
                'extreme_ratio','abusing_ratio','normal_ratio','unrewarded_ratio']
df_integrated = df_integrated.drop(columns=[c for c in cols_to_drop if c in df_integrated.columns])

df_integrated = df_integrated.merge(
    mda_score[['mda_idx','Total_Score','Risk_Label_score','Risk_Label',
               'extreme_ratio','abusing_ratio','normal_ratio','unrewarded_ratio']],
    on='mda_idx', how='left'
)
print(f"df_integrated 업데이트 완료: {df_integrated.shape}")
print(f"\n[row_label × Risk_Label 교차표]")
print(pd.crosstab(df_integrated['row_label'], df_integrated['Risk_Label']))

df_integrated 업데이트 완료: (16831054, 76)

[row_label × Risk_Label 교차표]
Risk_Label      경고     매우위험     위험      정상
row_label                                 
극단값          36806  6324546  24550    2000
미적립         629312   793817  29294  396649
어뷰징         242699  7468347  49153   22316
정상          475673     6600   6947  322345


## STEP 8. 손실액 정량화

비용은 전환(is_rewarded==1) 발생 시에만 청구됨. 미전환 클릭은 show_cost=0.
- **확정 손실**: 극단값 + 전환 발생 → 정산 반영 대상  
- **관리 손실**: 어뷰징 + 전환 발생 → 오염도 모니터링용

In [11]:
for c in ['show_cost', 'adv_cost', 'earn_cost', 'rwd_cost']:
    df_integrated[c] = df_integrated[c].fillna(0)

rewarded = df_integrated['is_rewarded'] == 1

total_show = df_integrated.loc[rewarded, 'show_cost'].sum()
total_earn = df_integrated.loc[rewarded, 'earn_cost'].sum()

confirmed_mask      = rewarded & (df_integrated['row_label'] == '극단값')
confirmed_show_loss = df_integrated.loc[confirmed_mask, 'show_cost'].sum()
confirmed_earn_loss = df_integrated.loc[confirmed_mask, 'earn_cost'].sum()

manageable_mask      = rewarded & (df_integrated['row_label'] == '어뷰징')
manageable_show_loss = df_integrated.loc[manageable_mask, 'show_cost'].sum()
manageable_earn_loss = df_integrated.loc[manageable_mask, 'earn_cost'].sum()

print(f"[전체 전환 비용]")
print(f"  광고주 지불(show_cost): {total_show:,.0f}원")
print(f"  매체 수취(earn_cost):   {total_earn:,.0f}원")
print(f"\n[확정 손실액: 극단값]")
print(f"  광고주 손실: {confirmed_show_loss:,.0f}원 ({confirmed_show_loss/total_show*100:.2f}%)")
print(f"  IVE 손실:    {confirmed_earn_loss:,.0f}원 ({confirmed_earn_loss/total_earn*100:.2f}%)")
print(f"\n[관리 손실액: 어뷰징]")
print(f"  광고주 손실: {manageable_show_loss:,.0f}원 ({manageable_show_loss/total_show*100:.2f}%)")
print(f"  IVE 손실:    {manageable_earn_loss:,.0f}원 ({manageable_earn_loss/total_earn*100:.2f}%)")

cost_by_label = (
    df_integrated[rewarded]
    .groupby('row_label')
    .agg(click_count=('click_key','count'), show_cost_sum=('show_cost','sum'), earn_cost_sum=('earn_cost','sum'))
    .reset_index()
)
cost_by_label['show_ratio(%)'] = (cost_by_label['show_cost_sum'] / total_show * 100).round(2)
cost_by_label['earn_ratio(%)'] = (cost_by_label['earn_cost_sum'] / total_earn * 100).round(2)
print(f"\n[row_label별 손실 현황]")
print(cost_by_label.sort_values('show_cost_sum', ascending=False).to_string(index=False))

cost_by_mda = (
    df_integrated[confirmed_mask]
    .groupby(['mda_idx', 'Risk_Label'])
    .agg(loss_show_cost=('show_cost','sum'), loss_earn_cost=('earn_cost','sum'),
         loss_clicks=('click_key','count'), rewarded_count=('is_rewarded','sum'),
         extreme_clicks=('row_label', lambda x: (x=='극단값').sum()),
         abusing_clicks=('row_label', lambda x: (x=='어뷰징').sum()))
    .reset_index()
    .sort_values('loss_show_cost', ascending=False)
)
cost_by_mda['loss_ratio'] = cost_by_mda['loss_show_cost'] / confirmed_show_loss * 100
print(f"\n[매체별 확정 손실 Top 15]")
print(cost_by_mda[['mda_idx','Risk_Label','loss_show_cost','loss_ratio','loss_clicks','rewarded_count','extreme_clicks','abusing_clicks']].head(15).to_string(index=False))

[전체 전환 비용]
  광고주 지불(show_cost): 577,199,458원
  매체 수취(earn_cost):   232,870,115원

[확정 손실액: 극단값]
  광고주 손실: 67,083,271원 (11.62%)
  IVE 손실:    55,246,949원 (23.72%)

[관리 손실액: 어뷰징]
  광고주 손실: 193,575,035원 (33.54%)
  IVE 손실:    27,365,703원 (11.75%)

[row_label별 손실 현황]
row_label  click_count  show_cost_sum  earn_cost_sum  show_ratio(%)  earn_ratio(%)
       정상       811565    316541152.0    150257463.0          54.84          64.52
      어뷰징       128177    193575035.0     27365703.0          33.54          11.75
      극단값       530791     67083271.0     55246949.0          11.62          23.72

[매체별 확정 손실 Top 15]
 mda_idx Risk_Label  loss_show_cost  loss_ratio  loss_clicks  rewarded_count  extreme_clicks  abusing_clicks
      58       매우위험      32527332.0   48.487993       148809          148809          148809               0
     563       매우위험      18149774.0   27.055589       310629          310629          310629               0
     342         경고       3170140.0    4.725679        11358

## STEP 9. CVR 분석 — 정상 vs 위험 매체 비교

In [12]:
cvr_comparison = (
    df_integrated
    .groupby('Risk_Label')
    .agg(total_clicks=('click_key','count'), total_rewarded=('is_rewarded','sum'),
         total_show=('show_cost','sum'), total_earn=('earn_cost','sum'))
    .reset_index()
)
cvr_comparison['CVR'] = (cvr_comparison['total_rewarded'] / cvr_comparison['total_clicks'] * 100).round(2)

print(f"[Risk_Label별 CVR 비교]")
print(cvr_comparison[['Risk_Label','total_clicks','total_rewarded','CVR','total_show','total_earn']]
      .sort_values('CVR', ascending=False).to_string(index=False))

cvr_normal = cvr_comparison[cvr_comparison['Risk_Label']=='정상']['CVR'].values[0]
cvr_danger = cvr_comparison[cvr_comparison['Risk_Label']=='매우위험']['CVR'].values
if len(cvr_danger) > 0:
    print(f"\n  정상 매체 CVR    : {cvr_normal:.2f}%")
    print(f"  매우위험 매체 CVR: {cvr_danger[0]:.2f}%")
    print(f"  → 정상 매체가 매우위험 대비 {cvr_normal/cvr_danger[0]:.1f}배 효율")

# 기회비용 분석
danger_clicks   = cvr_comparison[cvr_comparison['Risk_Label']=='매우위험']['total_clicks'].values[0]
danger_rewarded = cvr_comparison[cvr_comparison['Risk_Label']=='매우위험']['total_rewarded'].values[0]
expected  = danger_clicks * (cvr_normal / 100)
opp_loss  = expected - danger_rewarded

print(f"\n[기회비용 분석]")
print(f"  매우위험 매체 실제 전환:    {danger_rewarded:,}건")
print(f"  정상 CVR 기준 예상 전환:    {expected:,.0f}건")
print(f"  기회 손실 전환 수:          {opp_loss:,.0f}건")

clean_df   = df_integrated[df_integrated['row_label'].isin(['정상', '미적립'])]
cvr_before = df_integrated['is_rewarded'].sum() / len(df_integrated) * 100
cvr_after  = clean_df['is_rewarded'].sum() / len(clean_df) * 100
print(f"\n[극단값 제거 전후 CVR 변화]")
print(f"  제거 전: {cvr_before:.2f}%  →  제거 후: {cvr_after:.2f}%  (개선 {cvr_after - cvr_before:+.2f}%p)")

[Risk_Label별 CVR 비교]
Risk_Label  total_clicks  total_rewarded   CVR  total_show  total_earn
        정상        743310          332560 44.74  72521495.0  39464911.0
        경고       1384490          601253 43.43 435332148.0 136656086.0
        위험        109944           31325 28.49   7049094.0   5253319.0
      매우위험      14593310          505395  3.46  62296721.0  51495799.0

  정상 매체 CVR    : 44.74%
  매우위험 매체 CVR: 3.46%
  → 정상 매체가 매우위험 대비 12.9배 효율

[기회비용 분석]
  매우위험 매체 실제 전환:    505,395건
  정상 CVR 기준 예상 전환:    6,529,047건
  기회 손실 전환 수:          6,023,652건

[극단값 제거 전후 CVR 변화]
  제거 전: 8.74%  →  제거 후: 30.50%  (개선 +21.77%p)


## STEP 10. 시간대별 어뷰징 패턴

In [13]:
time_label = (
    df_integrated.groupby(['click_time', 'row_label'])
    .agg(click_count=('click_key','count')).reset_index()
)
time_total = df_integrated.groupby('click_time').agg(total_count=('click_key','count')).reset_index()
time_label = time_label.merge(time_total, on='click_time', how='left')
time_label['ratio'] = time_label['click_count'] / time_label['total_count'] * 100

print("[시간대별 row_label 비율 (%)]")
print(time_label.pivot_table(index='click_time', columns='row_label', values='ratio', fill_value=0).round(1).to_string())

def time_zone(h):
    if   0 <= h <= 5:  return '새벽(0~5시)'
    elif 6 <= h <= 8:  return '아침(6~8시)'
    elif 9 <= h <= 18: return '주간(9~18시)'
    else:              return '저녁(19~23시)'

df_integrated['time_zone'] = df_integrated['click_time'].apply(time_zone)

zone_label = df_integrated.groupby(['time_zone','row_label']).agg(click_count=('click_key','count')).reset_index()
zone_total = df_integrated.groupby('time_zone').agg(total_count=('click_key','count')).reset_index()
zone_label = zone_label.merge(zone_total, on='time_zone', how='left')
zone_label['ratio'] = zone_label['click_count'] / zone_label['total_count'] * 100

print("\n[시간대 구간별 row_label 비율 (%)]")
print(zone_label.pivot_table(index='time_zone', columns='row_label', values='ratio', fill_value=0).round(1).to_string())

[시간대별 row_label 비율 (%)]
row_label    극단값   미적립   어뷰징    정상
click_time                        
0           32.5  10.2  50.4   6.8
1           32.9  12.4  46.0   8.8
2           32.1  12.8  45.1  10.0
3           31.1  14.3  42.2  12.4
4           32.1  14.4  41.3  12.2
5           35.3  14.4  38.7  11.5
6           41.4  12.8  36.3   9.5
7           43.0  12.5  37.5   7.0
8           42.9  11.8  39.8   5.5
9           40.9  12.7  40.9   5.4
10          41.6  11.6  43.0   3.8
11          40.8  12.0  43.0   4.2
12          42.7  12.1  41.2   4.0
13          40.4  11.6  44.4   3.6
14          40.6  11.1  45.1   3.2
15          41.0  10.8  45.3   2.9
16          40.8  10.4  46.0   2.8
17          39.8  10.3  47.2   2.8
18          39.7  10.5  46.2   3.6
19          38.3   9.7  48.6   3.4
20          37.4   9.1  50.5   3.0
21          35.3   9.0  52.8   2.9
22          33.8   9.3  53.7   3.2
23          32.1   9.4  55.3   3.2

[시간대 구간별 row_label 비율 (%)]
row_label    극단값   미적립   어뷰징   정상
time

## STEP 11. 광고 유형별 어뷰징 취약성 + CVR 정화 효과

In [14]:
type_total = (
    df_integrated.groupby('ads_type_name')
    .agg(total_count=('click_key','count'), rewarded_count=('is_rewarded','sum')).reset_index()
)
type_total['cvr'] = (type_total['rewarded_count'] / type_total['total_count'] * 100).round(2)

type_clean = (
    df_integrated[df_integrated['row_label'].isin(['정상','미적립'])]
    .groupby('ads_type_name')
    .agg(clean_clicks=('click_key','count'), clean_rewarded=('is_rewarded','sum')).reset_index()
)
type_clean['clean_cvr'] = (type_clean['clean_rewarded'] / type_clean['clean_clicks'] * 100).round(2)

type_compare = type_total[['ads_type_name','total_count','cvr']].merge(
    type_clean[['ads_type_name','clean_clicks','clean_cvr']], on='ads_type_name', how='left'
)
type_compare['cvr_diff'] = (type_compare['clean_cvr'] - type_compare['cvr']).round(2)

print("[광고 유형별 CVR - 전체 vs 극단값 제거 후]")
print(type_compare.sort_values('cvr_diff', ascending=False).to_string(index=False))

[광고 유형별 CVR - 전체 vs 극단값 제거 후]
ads_type_name  total_count   cvr  clean_clicks  clean_cvr  cvr_diff
          설치형      1057472 55.37        505387      62.77      7.40
        인스타그램        18274 67.10         16279      69.05      1.95
          참여형     14391365  2.20       1018009       3.86      1.66
          유튜브        16814  6.11         12474       6.27      0.16
          클릭형         5336 98.58          3076      98.18     -0.40
          네이버        35524 54.48         33926      53.99     -0.49
          실행형      1280351 40.84       1053613      39.64     -1.20
         페이스북         4209 29.91          3868      28.08     -1.83
      CPS(구매)        21709 28.33         14005      20.79     -7.54


## STEP 12. 어뷰징 기기 블랙리스트 저장

In [15]:
device_stats = (
    df_integrated[df_integrated['row_label'] == '극단값']
    .groupby('dvc_idx')
    .agg(
        extreme_clicks=('click_key','count'),
        mda_count=('mda_idx','nunique'),
        ads_count=('ads_idx','nunique'),
        ip_count=('user_ip','nunique')
    )
    .reset_index()
    .sort_values('extreme_clicks', ascending=False)
)

total_extreme = len(df_integrated[df_integrated['row_label'] == '극단값'])
device_stats['click_ratio'] = device_stats['extreme_clicks'] / total_extreme * 100

print("[어뷰징 기기 블랙리스트 Top 20]")
print(device_stats[['dvc_idx','extreme_clicks','click_ratio','mda_count','ads_count','ip_count']].head(20).to_string(index=False))

for n in [10, 50, 100, 500]:
    top_ratio = device_stats.head(n)['extreme_clicks'].sum() / total_extreme * 100
    print(f"  상위 {n:>4}개 기기 → 극단값 클릭의 {top_ratio:.1f}% 차지")

device_stats.to_csv('/Users/joshuakim/Desktop/스파르타 최종데이터/device_blacklist.csv', index=False, encoding='utf-8-sig')
print(f"\n✅ 기기 블랙리스트 저장 완료: {len(device_stats):,}개 기기")

[어뷰징 기기 블랙리스트 Top 20]
 dvc_idx  extreme_clicks  click_ratio  mda_count  ads_count  ip_count
61747080           20346     0.318508          2         41         1
61515716            4820     0.075455          1          9        13
61900408            4255     0.066610          1          9         1
61916606            4116     0.064434          1          8         5
47353280            3979     0.062290          2         10        30
34514026            3703     0.057969          1       1991         1
61894179            3615     0.056591          1          8         1
29275033            3469     0.054306          1          9        37
29762799            3332     0.052161          2        359        10
32134636            3325     0.052052          1          9        36
35295361            3293     0.051551          1          9         1
54479014            3291     0.051519          1       1017         1
48118104            3261     0.051050          1       1465         

## STEP 13. df_report 전처리 (1년치)

집계 리포트이므로 비용=0인 시간대(전환 없는 구간)는 정상 데이터로 유지. 비용 구조 위반 행만 제거.

In [16]:
before = len(df_report)
df_report = df_report[
    (df_report['rpt_time_scost'] >= df_report['rpt_time_acost']) &
    (df_report['rpt_time_acost'] >= df_report['rpt_time_earn']) &
    (df_report['rpt_time_earn']  >= df_report['rpt_time_cost'])
].reset_index(drop=True)
print(f"df_report 이상치 제거: {before:,} → {len(df_report):,} (제거 {before - len(df_report):,}건)")

df_report['rpt_time_date'] = pd.to_datetime(df_report['rpt_time_date'], errors='coerce')
df_report['year_month']    = df_report['rpt_time_date'].dt.to_period('M')
df_report['weekday']       = df_report['rpt_time_date'].dt.day_name()

report_master = df_report.merge(
    df_master2[['ads_idx','domain_label','ads_type','ads_type_name','domain_name']],
    on='ads_idx', how='left'
)
report_master['domain_name'] = report_master['domain_name'].fillna('미분류')

print(f"\n[df_report 분석 기간]")
print(f"  {df_report['rpt_time_date'].min().date()} ~ {df_report['rpt_time_date'].max().date()}")
print(f"  총 {df_report['year_month'].nunique()}개월치")

# 1달 분석의 위험 매체 목록을 1년 리포트에 적용
danger_mda_list  = mda_score[mda_score['Risk_Label'] == '매우위험']['mda_idx'].tolist()
danger2_mda_list = mda_score[mda_score['Risk_Label'] == '위험']['mda_idx'].tolist()
warning_mda_list = mda_score[mda_score['Risk_Label'] == '경고']['mda_idx'].tolist()
normal_mda_list  = mda_score[mda_score['Risk_Label'] == '정상']['mda_idx'].tolist()

report_master['risk_group'] = np.select(
    [report_master['mda_idx'].isin(danger_mda_list),
     report_master['mda_idx'].isin(danger2_mda_list),
     report_master['mda_idx'].isin(warning_mda_list),
     report_master['mda_idx'].isin(normal_mda_list)],
    ['매우위험', '위험', '경고', '정상'], default='미분류'
)
print(f"\n[risk_group 분포]\n{report_master['risk_group'].value_counts()}")

df_report 이상치 제거: 6,953,146 → 6,892,108 (제거 61,038건)

[df_report 분석 기간]
  2024-07-27 ~ 2025-08-29
  총 14개월치

[risk_group 분포]
risk_group
매우위험    2514838
정상      1813240
경고      1013775
미분류      868451
위험       681804
Name: count, dtype: int64


## STEP 14. 월별 전체 CVR 추이 — 클릭 vs 전환 괴리

`gap = 클릭증가율 - 전환증가율`: 양수가 클수록 어뷰징 의심.

In [17]:
monthly_total = (
    report_master.groupby('year_month')
    .agg(
        total_clk   = ('rpt_time_clk',   'sum'),
        total_turn  = ('rpt_time_turn',  'sum'),
        total_scost = ('rpt_time_scost', 'sum'),
        total_acost = ('rpt_time_acost', 'sum'),
        total_earn  = ('rpt_time_earn',  'sum'),
    )
    .reset_index()
)
monthly_total['cvr']     = (monthly_total['total_turn'] / monthly_total['total_clk'] * 100).round(2)
monthly_total['clk_mom'] = monthly_total['total_clk'].pct_change().mul(100).round(2)
monthly_total['turn_mom']= monthly_total['total_turn'].pct_change().mul(100).round(2)
monthly_total['gap']     = (monthly_total['clk_mom'] - monthly_total['turn_mom']).round(2)

print("[월별 전체 CVR 추이 + 클릭/전환 괴리 (1년)]")
print(monthly_total[['year_month','total_clk','total_turn','cvr','clk_mom','turn_mom','gap','total_scost','total_earn']]
      .to_string(index=False))

[월별 전체 CVR 추이 + 클릭/전환 괴리 (1년)]
year_month  total_clk  total_turn   cvr  clk_mom  turn_mom    gap  total_scost  total_earn
   2024-07    1371671      663104 48.34      NaN       NaN    NaN     65436087    49430716
   2024-08    8519921     4432660 52.03   521.13    568.47 -47.34    523853663   409826806
   2024-09    6335492     3081024 48.63   -25.64    -30.49   4.85    478534887   370452029
   2024-10    5280601     2461174 46.61   -16.65    -20.12   3.47    374820578   283976418
   2024-11    5080758     2534906 49.89    -3.78      3.00  -6.78    424398456   324149488
   2024-12    4732724     1958877 41.39    -6.85    -22.72  15.87    386593815   283749002
   2025-01    4752333     2146887 45.18     0.41      9.60  -9.19    411042377   322632021
   2025-02    3123101     1083563 34.70   -34.28    -49.53  15.25    193059418   145655622
   2025-03    6796940     2965677 43.63   117.63    173.70 -56.07    528991434   411280561
   2025-04    3877259     1306576 33.70   -42.96    -55.94 

## STEP 15. mda 539 침투 타임라인

In [18]:
monthly_539 = (
    report_master[report_master['mda_idx'] == 539]
    .groupby('year_month')
    .agg(clk_539=('rpt_time_clk','sum'), turn_539=('rpt_time_turn','sum'), earn_539=('rpt_time_earn','sum'))
    .reset_index()
)
monthly_539 = monthly_total[['year_month','total_clk','cvr']].merge(monthly_539, on='year_month', how='left').fillna(0)
monthly_539['clk_share'] = (monthly_539['clk_539'] / monthly_539['total_clk'] * 100).round(2)
monthly_539['cvr_539']   = np.where(
    monthly_539['clk_539'] > 0,
    (monthly_539['turn_539'] / monthly_539['clk_539'] * 100).round(2), 0
)
monthly_539['cvr_gap'] = (monthly_539['cvr'] - monthly_539['cvr_539']).round(2)

print("[mda 539 월별 침투 타임라인]")
print(monthly_539[['year_month','total_clk','clk_539','clk_share','turn_539','cvr_539','cvr','cvr_gap','earn_539']].to_string(index=False))

first_539 = monthly_539[monthly_539['clk_539'] > 0]['year_month'].min()
peak_539  = monthly_539.loc[monthly_539['clk_share'].idxmax(), 'year_month']
print(f"\n  최초 등장: {first_539}  |  점유율 최고: {peak_539} ({monthly_539['clk_share'].max():.2f}%)")
print(f"  539 누적 earn_cost: {monthly_539['earn_539'].sum():,.0f}원")

[mda 539 월별 침투 타임라인]
year_month  total_clk  clk_539  clk_share  turn_539  cvr_539   cvr  cvr_gap  earn_539
   2024-07    1371671     8346       0.61      2785    33.37 48.34    14.97   1531650
   2024-08    8519921    46682       0.55     16569    35.49 52.03    16.54   7225850
   2024-09    6335492    60661       0.96     21675    35.73 48.63    12.90   5819450
   2024-10    5280601    41662       0.79     15690    37.66 46.61     8.95   6198020
   2024-11    5080758    28575       0.56     12068    42.23 49.89     7.66   3748880
   2024-12    4732724   194292       4.11     14721     7.58 41.39    33.81   5402430
   2025-01    4752333    71213       1.50     24037    33.75 45.18    11.43   8555820
   2025-02    3123101    22014       0.70      3636    16.52 34.70    18.18   3239870
   2025-03    6796940    74650       1.10     27989    37.49 43.63     6.14   6124560
   2025-04    3877259    30507       0.79      7835    25.68 33.70     8.02   4500260
   2025-05    4013274    26373   

## STEP 16. 위험 그룹별 월별 earn_cost & CVR 추이

In [19]:
monthly_risk = (
    report_master.groupby(['year_month','risk_group'])
    .agg(total_earn=('rpt_time_earn','sum'), total_clk=('rpt_time_clk','sum'), total_turn=('rpt_time_turn','sum'))
    .reset_index()
)
monthly_risk['cvr'] = (monthly_risk['total_turn'] / monthly_risk['total_clk'] * 100).round(2)

group_order    = ['매우위험','위험','경고','정상','미분류']
existing_groups= [g for g in group_order if g in monthly_risk['risk_group'].unique()]

print("[위험 그룹별 월별 earn_cost 추이]")
print(monthly_risk.pivot_table(index='year_month', columns='risk_group', values='total_earn', fill_value=0)[existing_groups].round(0).to_string())
print("\n[위험 그룹별 월별 CVR 추이]")
print(monthly_risk.pivot_table(index='year_month', columns='risk_group', values='cvr', fill_value=0)[existing_groups].round(2).to_string())

summary = monthly_risk.groupby('risk_group').agg(누적_클릭=('total_clk','sum'), 누적_전환=('total_turn','sum'), 누적_earn=('total_earn','sum')).reset_index()
summary['평균_CVR'] = (summary['누적_전환'] / summary['누적_클릭'] * 100).round(2)
summary['earn_비율']= (summary['누적_earn'] / summary['누적_earn'].sum() * 100).round(2)
print("\n[위험 그룹별 1년 누적 요약]")
print(summary[['risk_group','누적_클릭','누적_전환','평균_CVR','누적_earn','earn_비율']].set_index('risk_group').loc[existing_groups].to_string())

[위험 그룹별 월별 earn_cost 추이]
risk_group         매우위험          위험           경고          정상         미분류
year_month                                                              
2024-07      14678921.0   1979825.0   23496857.0   7498680.0   1776433.0
2024-08     133156799.0  12124604.0  200732945.0  53916230.0   9896228.0
2024-09     153638996.0   7964800.0  156826946.0  40805800.0  11215487.0
2024-10      81271671.0   8035004.0  145267257.0  30730135.0  18672351.0
2024-11      76463801.0  19523486.0  158134973.0  45597219.0  24430009.0
2024-12      81662021.0  11081557.0  137903621.0  33021098.0  20080705.0
2025-01     110481468.0   7936362.0  153015981.0  36714155.0  14484055.0
2025-02      41936960.0   5405409.0   63080297.0  19970402.0  15262554.0
2025-03     114659890.0   8050443.0  222287269.0  51734153.0  14548806.0
2025-04      41268349.0   5091629.0   76567571.0  25780030.0  13210821.0
2025-05      49224293.0   6470970.0   72036638.0  30074332.0  15075782.0
2025-06      34435558.0   

## STEP 17. 요일별 패턴 — 정상 vs 매우위험 비교 (봇 증거)

In [20]:
weekday_order = ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']
weekday_kor   = dict(zip(weekday_order, ['월','화','수','목','금','토','일']))

weekday_normal = (
    report_master[report_master['risk_group']=='정상']
    .groupby('weekday').agg(avg_clk_normal=('rpt_time_clk','mean'), avg_turn_normal=('rpt_time_turn','mean'))
    .reindex(weekday_order).reset_index()
)
weekday_normal['cvr_normal'] = (weekday_normal['avg_turn_normal'] / weekday_normal['avg_clk_normal'] * 100).round(2)

weekday_danger = (
    report_master[report_master['risk_group']=='매우위험']
    .groupby('weekday').agg(avg_clk_danger=('rpt_time_clk','mean'), avg_turn_danger=('rpt_time_turn','mean'))
    .reindex(weekday_order).reset_index()
)
weekday_danger['cvr_danger'] = (weekday_danger['avg_turn_danger'] / weekday_danger['avg_clk_danger'] * 100).round(2)

weekday_compare = weekday_normal.merge(weekday_danger, on='weekday', how='left')
weekday_compare['weekday_kor'] = weekday_compare['weekday'].map(weekday_kor)

print("[요일별 평균 클릭 패턴 비교 (정상 vs 매우위험)]")
print(weekday_compare[['weekday_kor','avg_clk_normal','cvr_normal','avg_clk_danger','cvr_danger']].to_string(index=False))

def week_avg(df, col, days):
    return df[df['weekday'].isin(days)][col].mean()

weekdays = ['Monday','Tuesday','Wednesday','Thursday','Friday']
weekends = ['Saturday','Sunday']

for label, clk_col, cvr_col in [('정상','avg_clk_normal','cvr_normal'),('매우위험','avg_clk_danger','cvr_danger')]:
    wkd_clk  = week_avg(weekday_compare, clk_col, weekdays)
    wknd_clk = week_avg(weekday_compare, clk_col, weekends)
    wkd_cvr  = week_avg(weekday_compare, cvr_col, weekdays)
    wknd_cvr = week_avg(weekday_compare, cvr_col, weekends)
    print(f"  {label:6}: 주중 {wkd_clk:.1f} / 주말 {wknd_clk:.1f} ({(wknd_clk-wkd_clk)/wkd_clk*100:+.1f}%)  "
          f"| CVR 주중 {wkd_cvr:.2f}% / 주말 {wknd_cvr:.2f}% ({wknd_cvr-wkd_cvr:+.2f}%p)")

[요일별 평균 클릭 패턴 비교 (정상 vs 매우위험)]
weekday_kor  avg_clk_normal  cvr_normal  avg_clk_danger  cvr_danger
          월        5.138577       38.62       13.481006       34.40
          화        5.804243       38.69       10.246195       52.11
          수        5.875148       40.23       13.788380       40.53
          목        6.524165       43.08       18.441532       36.13
          금        6.319219       45.01       20.242558       33.41
          토        5.623398       41.85       18.058846       30.15
          일        5.318981       38.85       17.718136       26.67
  정상    : 주중 5.9 / 주말 5.5 (-7.8%)  | CVR 주중 41.13% / 주말 40.35% (-0.78%p)
  매우위험  : 주중 15.2 / 주말 17.9 (+17.4%)  | CVR 주중 39.32% / 주말 28.41% (-10.91%p)


## STEP 18. 광고유형별 + 도메인별 1년 트렌드

In [21]:
key_types = ['참여형','설치형','클릭형','인스타그램','CPS(구매)','실행형']

monthly_type = (
    report_master.groupby(['year_month','ads_type_name'])
    .agg(total_clk=('rpt_time_clk','sum'), total_turn=('rpt_time_turn','sum')).reset_index()
)
monthly_type['cvr'] = (monthly_type['total_turn'] / monthly_type['total_clk'] * 100).round(2)

print("[주요 광고유형별 월별 CVR 추이]")
print(monthly_type[monthly_type['ads_type_name'].isin(key_types)]
      .pivot_table(index='year_month', columns='ads_type_name', values='cvr', fill_value=0).round(2).to_string())

monthly_domain = (
    report_master.groupby(['year_month','domain_name'])
    .agg(total_clk=('rpt_time_clk','sum'), total_turn=('rpt_time_turn','sum'), total_scost=('rpt_time_scost','sum')).reset_index()
)
monthly_domain['cvr'] = (monthly_domain['total_turn'] / monthly_domain['total_clk'] * 100).round(2)

domain_clk_pivot = monthly_domain.pivot_table(index='year_month', columns='domain_name', values='total_clk', fill_value=0)
domain_clk_share = domain_clk_pivot.div(domain_clk_pivot.sum(axis=1), axis=0).mul(100).round(2)
print("\n[도메인별 월별 클릭 점유율 % (1년)]")
print(domain_clk_share.to_string())
print("\n[도메인별 월별 CVR 추이]")
print(monthly_domain.pivot_table(index='year_month', columns='domain_name', values='cvr', fill_value=0).round(2).to_string())

[주요 광고유형별 월별 CVR 추이]
ads_type_name  CPS(구매)    설치형    실행형  인스타그램    참여형     클릭형
year_month                                                
2024-07           0.52  41.30  37.08  33.33  54.28    0.00
2024-08           1.23  53.20  43.76  24.68  54.62    0.00
2024-09           1.28  55.43  48.09  46.58  44.18   97.66
2024-10           1.78  51.83  44.87  32.38  45.31    0.00
2024-11           1.16  52.49  47.89  41.42  50.68    0.00
2024-12           0.56  55.59  32.38  38.95  35.76   99.01
2025-01           1.80  54.14  45.68  53.61  35.44   99.32
2025-02           3.43  50.69  24.49  30.35  33.36   98.84
2025-03           6.70  54.55  36.71  42.35  40.56   99.66
2025-04          11.40  52.31  22.78  29.29  36.71   96.35
2025-05           6.98  57.36  24.23  56.46  43.56   88.85
2025-06          11.64  46.98  24.52  63.98  51.07  100.00
2025-07          28.18  52.96  29.68  84.95  50.86   99.68
2025-08          22.75  54.31  45.39  64.42   1.77   98.55

[도메인별 월별 클릭 점유율 % (1년)]
domain_nam

## STEP 19. 1달 vs 1년 비교 검증

In [22]:
one_month_ym   = report_master['year_month'].max()
om_row         = monthly_total[monthly_total['year_month'] == one_month_ym]
om_cvr         = om_row['cvr'].values[0]
om_clk         = om_row['total_clk'].values[0]
om_539         = monthly_539[monthly_539['year_month'] == one_month_ym]['clk_share'].values
om_539         = om_539[0] if len(om_539) > 0 else 0

yearly_avg_cvr = monthly_total['cvr'].mean().round(2)
yearly_avg_clk = monthly_total['total_clk'].mean()
cvr_z = (om_cvr - monthly_total['cvr'].mean()) / monthly_total['cvr'].std()
clk_z = (om_clk - monthly_total['total_clk'].mean()) / monthly_total['total_clk'].std()

print(f"[1달 vs 1년 비교 검증] 대상 월: {one_month_ym}")
print(f"  {'지표':15}  {'1년 월평균':>12}  {'1달 해당월':>12}  {'차이':>10}")
print(f"  {'CVR (%)':15}  {yearly_avg_cvr:>12.2f}  {om_cvr:>12.2f}  {om_cvr-yearly_avg_cvr:>+10.2f}")
print(f"  {'총 클릭수':15}  {yearly_avg_clk:>12,.0f}  {om_clk:>12,.0f}  {om_clk-yearly_avg_clk:>+10,.0f}")
print(f"  {'539 점유율(%)':15}  {'(월별 확인)':>12}  {om_539:>12.2f}")
print(f"\n  CVR  Z-score: {cvr_z:+.2f}  {'⚠️ 이상 구간' if abs(cvr_z)>1.5 else '정상 범위'}")
print(f"  클릭 Z-score: {clk_z:+.2f}  {'⚠️ 이상 구간' if abs(clk_z)>1.5 else '정상 범위'}")

[1달 vs 1년 비교 검증] 대상 월: 2025-08
  지표                     1년 월평균        1달 해당월          차이
  CVR (%)                 41.77          9.06      -32.71
  총 클릭수               5,533,167    16,889,772  +11,356,605
  539 점유율(%)            (월별 확인)         79.76

  CVR  Z-score: -3.04  ⚠️ 이상 구간
  클릭 Z-score: +3.06  ⚠️ 이상 구간


## CVR 하락 기여도 분석 — mda 58 vs mda 539

In [23]:
total_clk_aug  = len(df_integrated)
total_turn_aug = df_integrated['is_rewarded'].sum()
cvr_aug_total  = total_turn_aug / total_clk_aug * 100

print(f"【 8월 전체 기준 】  총 클릭: {total_clk_aug:,}  총 전환: {total_turn_aug:,}  CVR: {cvr_aug_total:.2f}%")

for mda_id in [58, 539]:
    mask       = df_integrated['mda_idx'] == mda_id
    clk        = mask.sum()
    turn       = df_integrated.loc[mask, 'is_rewarded'].sum()
    cvr        = turn / clk * 100 if clk > 0 else 0
    risk       = df_integrated.loc[mask, 'Risk_Label'].iloc[0] if clk > 0 else 'N/A'
    row_dist   = df_integrated.loc[mask, 'row_label'].value_counts()
    print(f"\n【 mda {mda_id} — {risk} 】  클릭: {clk:,}  전환: {turn:,}  CVR: {cvr:.2f}%  (전체 대비 {cvr-cvr_aug_total:+.2f}%p)")
    for label, cnt in row_dist.items():
        print(f"  {label:6}  {cnt:>8,}건  ({cnt/clk*100:.1f}%)")

print("\n【 핵심 — 매체 제거 시 CVR 변화 】")
for mda_id in [58, 539, (58, 539)]:
    if isinstance(mda_id, tuple):
        mask_remove = df_integrated['mda_idx'].isin(mda_id)
        label = "mda 58 + 539 동시 제거"
    else:
        mask_remove = df_integrated['mda_idx'] == mda_id
        label = f"mda {mda_id} 제거 시"
    df_wo        = df_integrated[~mask_remove]
    cvr_without  = df_wo['is_rewarded'].sum() / len(df_wo) * 100
    print(f"  {label}: 제거 후 CVR {cvr_without:.2f}%  (▲ +{cvr_without - cvr_aug_total:.2f}%p)")

total_wasted = total_clk_aug - total_turn_aug
print(f"\n【 낭비 클릭 기여도 】  전체 낭비: {total_wasted:,}건")
for mda_id in [58, 539]:
    mask  = df_integrated['mda_idx'] == mda_id
    clk   = mask.sum(); turn = df_integrated.loc[mask,'is_rewarded'].sum()
    waste = clk - turn
    print(f"  mda {mda_id}: 낭비 {waste:,}건  (전체 낭비의 {waste/total_wasted*100:.1f}%)")

print("\n【 CVR 기여도 Top 10 — 제거 시 CVR 회복폭 기준 】")
results = []
for mda_id in df_integrated['mda_idx'].unique():
    mask = df_integrated['mda_idx'] == mda_id
    clk  = mask.sum()
    if clk < 1000:
        continue
    turn     = df_integrated.loc[mask,'is_rewarded'].sum()
    df_wo    = df_integrated[~mask]
    cvr_wo   = df_wo['is_rewarded'].sum() / len(df_wo) * 100
    risk     = df_integrated.loc[mask,'Risk_Label'].iloc[0]
    results.append({'mda_idx': mda_id, 'Risk_Label': risk, 'clk': clk,
                    'clk_share_%': round(clk/total_clk_aug*100,2),
                    'mda_cvr_%': round(turn/clk*100,2),
                    'cvr_회복폭_%p': round(cvr_wo - cvr_aug_total, 3)})

rank_df = pd.DataFrame(results).sort_values('cvr_회복폭_%p', ascending=False).reset_index(drop=True)
print(rank_df[['mda_idx','Risk_Label','clk','clk_share_%','mda_cvr_%','cvr_회복폭_%p']].head(10).to_string(index=False))

【 8월 전체 기준 】  총 클릭: 16,831,054  총 전환: 1,470,533  CVR: 8.74%

【 mda 58 — 매우위험 】  클릭: 482,794  전환: 155,827  CVR: 32.28%  (전체 대비 +23.54%p)
  극단값      410,774건  (85.1%)
  어뷰징       69,457건  (14.4%)
  미적립        2,495건  (0.5%)
  정상            68건  (0.0%)

【 mda 539 — 매우위험 】  클릭: 13,468,019  전환: 9,179  CVR: 0.07%  (전체 대비 -8.67%p)
  어뷰징     7,397,726건  (54.9%)
  극단값     5,272,928건  (39.2%)
  미적립      791,010건  (5.9%)
  정상         6,355건  (0.0%)

【 핵심 — 매체 제거 시 CVR 변화 】
  mda 58 제거 시: 제거 후 CVR 8.04%  (▲ +-0.70%p)
  mda 539 제거 시: 제거 후 CVR 43.45%  (▲ +34.72%p)
  mda 58 + 539 동시 제거: 제거 후 CVR 45.33%  (▲ +36.59%p)

【 낭비 클릭 기여도 】  전체 낭비: 15,360,521건
  mda 58: 낭비 326,967건  (전체 낭비의 2.1%)
  mda 539: 낭비 13,458,840건  (전체 낭비의 87.6%)

【 CVR 기여도 Top 10 — 제거 시 CVR 회복폭 기준 】
 mda_idx Risk_Label      clk  clk_share_%  mda_cvr_%  cvr_회복폭_%p
     539       매우위험 13468019        80.02       0.07      34.716
     654         정상    24370         0.14       4.54       0.006
      18         정상     4937         0.03   

## 확정 손실 Top 3 매체

In [24]:
confirmed_all_mask   = (df_integrated['is_rewarded'] == 1) & (df_integrated['row_label'] == '극단값')
total_confirmed_loss = df_integrated.loc[confirmed_all_mask, 'show_cost'].sum()

loss_by_mda = (
    df_integrated[confirmed_all_mask]
    .groupby('mda_idx')
    .agg(confirmed_loss=('show_cost','sum'), extreme_clicks=('click_key','count'), Risk_Label=('Risk_Label','first'))
    .reset_index()
    .sort_values('confirmed_loss', ascending=False)
    .reset_index(drop=True)
)
loss_by_mda['loss_ratio_%'] = loss_by_mda['confirmed_loss'] / total_confirmed_loss * 100

print(f"전체 확정 손실 합계: {total_confirmed_loss:,.0f}원  ({total_confirmed_loss/1e4:.0f}만원)")
for i, row in loss_by_mda.head(3).iterrows():
    print(f"\n  {i+1}위  mda {int(row['mda_idx'])}  [{row['Risk_Label']}]")
    print(f"    확정 손실액: {row['confirmed_loss']:>15,.0f}원  ({row['confirmed_loss']/1e4:.0f}만원)")
    print(f"    전체 비중  : {row['loss_ratio_%']:.1f}%  |  극단값 클릭: {row['extreme_clicks']:,}건")

top3_loss  = loss_by_mda.head(3)['confirmed_loss'].sum()
print(f"\nTop 3 합산: {top3_loss:,.0f}원  ({top3_loss/total_confirmed_loss*100:.1f}%)")

전체 확정 손실 합계: 67,083,271원  (6708만원)

  1위  mda 58  [매우위험]
    확정 손실액:      32,527,332원  (3253만원)
    전체 비중  : 48.5%  |  극단값 클릭: 148,809건

  2위  mda 563  [매우위험]
    확정 손실액:      18,149,774원  (1815만원)
    전체 비중  : 27.1%  |  극단값 클릭: 310,629건

  3위  mda 342  [경고]
    확정 손실액:       3,170,140원  (317만원)
    전체 비중  : 4.7%  |  극단값 클릭: 11,358건

Top 3 합산: 53,847,246원  (80.3%)


## mda 539 슬라이드 수치 검증

In [25]:
MDA_ID = 539
mask_539   = df_integrated['mda_idx'] == MDA_ID
df_539     = df_integrated[mask_539]
clk_539    = len(df_539)
turn_539   = df_539['is_rewarded'].sum()
cvr_539    = turn_539 / clk_539 * 100 if clk_539 > 0 else 0
clk_share_539 = clk_539 / total_clk_aug * 100

df_without_539  = df_integrated[~mask_539]
cvr_without_539 = df_without_539['is_rewarded'].sum() / len(df_without_539) * 100

waste_539       = clk_539 - turn_539
waste_share_539 = waste_539 / (total_clk_aug - total_turn_aug) * 100

mask_58  = df_integrated['mda_idx'] == 58
df_58    = df_integrated[mask_58]
clk_58   = len(df_58); turn_58 = df_58['is_rewarded'].sum()
cvr_58   = turn_58 / clk_58 * 100

confirmed_539_mask = (df_integrated['is_rewarded']==1) & mask_539 & (df_integrated['row_label']=='극단값')
confirmed_58_mask  = (df_integrated['is_rewarded']==1) & mask_58  & (df_integrated['row_label']=='극단값')
loss_539 = df_integrated.loc[confirmed_539_mask, 'show_cost'].sum()
loss_58  = df_integrated.loc[confirmed_58_mask,  'show_cost'].sum()

er_539 = mda_score.loc[mda_score['mda_idx']==MDA_ID, 'extreme_ratio'].values
er_58  = mda_score.loc[mda_score['mda_idx']==58,     'extreme_ratio'].values

print(f"【 매체 프로필 】")
print(f"  8월 클릭 점유율 : {clk_share_539:.1f}%  ({clk_539:,}건)")
print(f"  매체 CVR        : {cvr_539:.2f}%  (전체 {cvr_aug_total:.2f}% 대비 {cvr_539-cvr_aug_total:+.2f}%p)")
print(f"  최초 등장       : {monthly_539[monthly_539['clk_539']>0]['year_month'].min()}")
print(f"  539 제거 시 CVR : {cvr_without_539:.2f}%  (+{cvr_without_539-cvr_aug_total:.2f}%p 회복)")
print(f"  낭비 클릭       : {waste_539:,}건  (전체 낭비의 {waste_share_539:.1f}%)")

print(f"\n【 mda 58 vs 539 비교 】")
print(f"  {'지표':<16}  {'mda 58':>14}  {'mda 539':>14}")
print(f"  {'-'*48}")
for row_label, v58, v539 in [
    ('클릭 수',      f'{clk_58:,}건',     f'{clk_539:,}건'),
    ('전환 수',      f'{turn_58:,}건',    f'{turn_539:,}건'),
    ('CVR',          f'{cvr_58:.2f}%',    f'{cvr_539:.2f}%'),
    ('확정 손실액',  f'{loss_58/1e4:.0f}만원', f'{loss_539/1e4:.0f}만원'),
    ('극단값 비율',  f'{er_58[0]*100:.1f}%' if len(er_58) else 'N/A', f'{er_539[0]*100:.1f}%' if len(er_539) else 'N/A'),
    ('낭비 클릭',    f'{(clk_58-turn_58):,}건', f'{waste_539:,}건'),
]:
    print(f"  {row_label:<16}  {v58:>14}  {v539:>14}")

# PHASE 구분
print(f"\n【 침투 타임라인 PHASE 】")
print(f"  PHASE 1 잠복: {monthly_539[monthly_539['clk_539']>0].iloc[0]['year_month']} ~ 25.05")
print(f"  PHASE 2 확장: 25.06 ~ 25.07")
print(f"  PHASE 3 폭발: 25.08  (점유율 {monthly_539['clk_share'].max():.1f}%, CVR {monthly_539.iloc[-1]['cvr']:.2f}%)")

【 매체 프로필 】
  8월 클릭 점유율 : 80.0%  (13,468,019건)
  매체 CVR        : 0.07%  (전체 8.74% 대비 -8.67%p)
  최초 등장       : 2024-07
  539 제거 시 CVR : 43.45%  (+34.72%p 회복)
  낭비 클릭       : 13,458,840건  (전체 낭비의 87.6%)

【 mda 58 vs 539 비교 】
  지표                        mda 58         mda 539
  ------------------------------------------------
  클릭 수                    482,794건     13,468,019건
  전환 수                    155,827건          9,179건
  CVR                       32.28%           0.07%
  확정 손실액                    3253만원            49만원
  극단값 비율                     85.1%           39.2%
  낭비 클릭                   326,967건     13,458,840건

【 침투 타임라인 PHASE 】
  PHASE 1 잠복: 2024-07 ~ 25.05
  PHASE 2 확장: 25.06 ~ 25.07
  PHASE 3 폭발: 25.08  (점유율 79.8%, CVR 9.06%)


## mda 58 집중 분석

In [26]:
MDA_TARGET = 58
mda58          = df_integrated[df_integrated['mda_idx'] == MDA_TARGET].copy()
mda58_rewarded = mda58[mda58['is_rewarded'] == 1]

total_clk       = len(mda58)
total_turn      = mda58['is_rewarded'].sum()
cvr_mda58       = total_turn / total_clk * 100
total_show_cost = mda58_rewarded['show_cost'].sum()
total_earn_cost = mda58_rewarded['earn_cost'].sum()
label_dist      = mda58['row_label'].value_counts()
label_pct       = (label_dist / total_clk * 100).round(2)

normal_mda = df_integrated[df_integrated['Risk_Label'] == '정상']
cvr_normal = normal_mda['is_rewarded'].sum() / len(normal_mda) * 100

print(f"[A. 기본 프로필]")
print(f"  클릭: {total_clk:,}  전환: {total_turn:,}  CVR: {cvr_mda58:.2f}%  (정상 {cvr_normal:.2f}% 대비 {cvr_mda58-cvr_normal:+.2f}%p)")
print(f"  show_cost: {total_show_cost:,.0f}원  earn_cost: {total_earn_cost:,.0f}원")
print("\n  [클릭 품질]")
for lbl in ['극단값','어뷰징','미적립','정상']:
    cnt = label_dist.get(lbl, 0); pct = label_pct.get(lbl, 0)
    print(f"    {lbl:5}: {cnt:>7,}건  {pct:>5.1f}%  {'█'*int(pct/2)}")

mda58_loss = mda58_rewarded[mda58_rewarded['row_label']=='극단값']['show_cost'].sum()
print(f"\n  확정 손실: {mda58_loss:,.0f}원  (전체의 {mda58_loss/total_confirmed_loss*100:.1f}%)")

[A. 기본 프로필]
  클릭: 482,794  전환: 155,827  CVR: 32.28%  (정상 44.74% 대비 -12.46%p)
  show_cost: 35,167,572원  earn_cost: 31,171,690원

  [클릭 품질]
    극단값  : 410,774건   85.1%  ██████████████████████████████████████████
    어뷰징  :  69,457건   14.4%  ███████
    미적립  :   2,495건    0.5%  
    정상   :      68건    0.0%  

  확정 손실: 32,527,332원  (전체의 48.5%)


In [27]:
# B. CTIT / IP 지표 분포 비교
ctit_58     = mda58.dropna(subset=['ctit'])['ctit']
ctit_normal = normal_mda.dropna(subset=['ctit'])['ctit']

print("[B. 탐지 지표 분포 (mda 58 vs 정상 매체)]")
print(f"  {'지표':20} {'mda 58':>12} {'정상 매체':>12}")
for stat, fn in [('중앙값(초)', 'median'), ('평균(초)', 'mean'),
                 ('p10', lambda x: x.quantile(0.1)), ('p90', lambda x: x.quantile(0.9))]:
    v58 = fn(ctit_58) if callable(fn) else getattr(ctit_58, fn)()
    vn  = fn(ctit_normal) if callable(fn) else getattr(ctit_normal, fn)()
    print(f"  {stat:20} {v58:>12.1f} {vn:>12.1f}")

ip_58 = ip_stats[ip_stats['mda_idx'] == MDA_TARGET]
print(f"\n  {'지표':20} {'mda 58':>12} {'전체 정상':>12}")
for col, label in [('device_count','IP당 기기수'),('click_count','IP당 클릭수')]:
    v58 = ip_58[col].median()
    vn  = ip_stats[ip_stats['mda_idx'].isin(normal_mda_list)][col].median()
    print(f"  {label:20} {v58:>12.1f} {vn:>12.1f}")

[B. 탐지 지표 분포 (mda 58 vs 정상 매체)]
  지표                         mda 58        정상 매체
  중앙값(초)                       71.0         95.0
  평균(초)                       450.4        409.3
  p10                           2.0         51.0
  p90                         256.0        199.0

  지표                         mda 58        전체 정상
  IP당 기기수                      10.0          1.0
  IP당 클릭수                      11.0          1.0


In [28]:
# C. 월별 트렌드 (1년)
mda58_monthly = (
    report_master[report_master['mda_idx']==MDA_TARGET]
    .groupby('year_month')
    .agg(clk_58=('rpt_time_clk','sum'), turn_58=('rpt_time_turn','sum'), earn_58=('rpt_time_earn','sum'))
    .reset_index()
)
mda58_monthly = monthly_total[['year_month','total_clk','total_turn','cvr']].merge(mda58_monthly, on='year_month', how='left').fillna(0)
mda58_monthly['cvr_58']    = np.where(mda58_monthly['clk_58']>0, (mda58_monthly['turn_58']/mda58_monthly['clk_58']*100).round(2), 0)
mda58_monthly['clk_share'] = (mda58_monthly['clk_58']/mda58_monthly['total_clk']*100).round(2)
mda58_monthly['cvr_gap']   = (mda58_monthly['cvr']-mda58_monthly['cvr_58']).round(2)

print("[C. 월별 트렌드 (1년)]")
for _, r in mda58_monthly.iterrows():
    marker = ' ◀ 이상' if r['clk_share'] > 5 else ''
    print(f"  {str(r['year_month']):>8} | 전체:{r['total_clk']:>10,.0f} | 58:{r['clk_58']:>10,.0f} ({r['clk_share']:.2f}%) | CVR 58:{r['cvr_58']:.2f}% 전체:{r['cvr']:.2f}%{marker}")

print(f"\n  최초 등장: {mda58_monthly[mda58_monthly['clk_58']>0]['year_month'].min()}")
print(f"  클릭 최고: {mda58_monthly.loc[mda58_monthly['clk_58'].idxmax(),'year_month']}  ({mda58_monthly['clk_58'].max():,.0f}건)")
print(f"  1년 누적 earn: {mda58_monthly['earn_58'].sum():,.0f}원")

[C. 월별 트렌드 (1년)]
   2024-07 | 전체: 1,371,671 | 58:    69,193 (5.04%) | CVR 58:35.90% 전체:48.34% ◀ 이상
   2024-08 | 전체: 8,519,921 | 58:   758,384 (8.90%) | CVR 58:45.21% 전체:52.03% ◀ 이상
   2024-09 | 전체: 6,335,492 | 58: 1,161,251 (18.33%) | CVR 58:52.46% 전체:48.63% ◀ 이상
   2024-10 | 전체: 5,280,601 | 58:   463,998 (8.79%) | CVR 58:46.55% 전체:46.61% ◀ 이상
   2024-11 | 전체: 5,080,758 | 58:   492,236 (9.69%) | CVR 58:40.35% 전체:49.89% ◀ 이상
   2024-12 | 전체: 4,732,724 | 58:   518,409 (10.95%) | CVR 58:43.15% 전체:41.39% ◀ 이상
   2025-01 | 전체: 4,752,333 | 58:   716,359 (15.07%) | CVR 58:45.92% 전체:45.18% ◀ 이상
   2025-02 | 전체: 3,123,101 | 58:   322,022 (10.31%) | CVR 58:26.32% 전체:34.70% ◀ 이상
   2025-03 | 전체: 6,796,940 | 58:   855,574 (12.59%) | CVR 58:44.99% 전체:43.63% ◀ 이상
   2025-04 | 전체: 3,877,259 | 58:   270,593 (6.98%) | CVR 58:30.56% 전체:33.70% ◀ 이상
   2025-05 | 전체: 4,013,274 | 58:   245,685 (6.12%) | CVR 58:44.67% 전체:41.91% ◀ 이상
   2025-06 | 전체: 3,202,453 | 58:   123,470 (3.86%) | CVR 58:12.19% 전체:44.63%

In [29]:
# D. 시간대별 패턴
time_58 = mda58.groupby(['click_time','row_label']).agg(cnt=('click_key','count')).reset_index()
time_58_total = mda58.groupby('click_time')['click_key'].count().reset_index(name='total')
time_58 = time_58.merge(time_58_total, on='click_time')
time_58['pct'] = (time_58['cnt']/time_58['total']*100).round(1)

pivot_58   = time_58.pivot_table(index='click_time', columns='row_label', values='pct', fill_value=0).reset_index()
time_norm  = normal_mda.groupby(['click_time','row_label']).agg(cnt=('click_key','count')).reset_index()
time_norm_t= normal_mda.groupby('click_time')['click_key'].count().reset_index(name='total')
time_norm  = time_norm.merge(time_norm_t, on='click_time')
time_norm['pct'] = (time_norm['cnt']/time_norm['total']*100).round(1)
pivot_norm = time_norm.pivot_table(index='click_time', columns='row_label', values='pct', fill_value=0).reset_index()

compare_time = pivot_58[['click_time']].copy()
for col in ['극단값','정상']:
    compare_time[f'58_{col}%']       = pivot_58.get(col, 0)
    compare_time[f'정상매체_{col}%'] = pivot_norm.get(col, 0)

print("[D. 시간대별 패턴 (mda 58 vs 정상 매체)]")
print(compare_time.to_string(index=False))

dawn    = pivot_58[pivot_58['click_time'].between(0,5)].get('극단값', pd.Series([0])).mean()
daytime = pivot_58[pivot_58['click_time'].between(9,18)].get('극단값', pd.Series([0])).mean()
print(f"\n  새벽(0~5시) 극단값 평균: {dawn:.1f}%  |  주간(9~18시): {daytime:.1f}%  |  차이: {dawn-daytime:+.1f}%p")

[D. 시간대별 패턴 (mda 58 vs 정상 매체)]
 click_time  58_극단값%  정상매체_극단값%  58_정상%  정상매체_정상%
          0     86.1        0.4     0.0      56.2
          1     93.0        0.3     0.0      58.9
          2     95.5        0.2     0.0      59.6
          3     96.5        0.1     0.0      60.2
          4     95.3        0.1     0.0      58.0
          5     92.5        0.1     0.0      55.8
          6     85.9        0.1     0.0      53.7
          7     85.6        0.1     0.0      46.6
          8     82.4        0.1     0.0      44.2
          9     80.2        0.2     0.0      42.2
         10     76.9        0.3     0.0      36.9
         11     85.1        0.3     0.0      36.2
         12     89.8        0.3     0.0      34.6
         13     87.4        0.2     0.0      32.2
         14     83.6        0.3     0.0      28.7
         15     80.1        0.2     0.0      30.0
         16     78.1        0.2     0.0      31.1
         17     75.5        0.2     0.0      29.7
         18     80.

In [30]:
# E. 요일별 패턴
weekday_kor_map = {'Monday':'월','Tuesday':'화','Wednesday':'수','Thursday':'목','Friday':'금','Saturday':'토','Sunday':'일'}

if 'weekday' in mda58.columns:
    wd_58 = mda58.groupby('weekday').agg(clk=('click_key','count'),turn=('is_rewarded','sum')).reindex(weekday_order).reset_index()
    wd_58['cvr'] = (wd_58['turn']/wd_58['clk']*100).round(2)
    wd_58['kor'] = wd_58['weekday'].map(weekday_kor_map)
    wd_norm = normal_mda.groupby('weekday').agg(clk=('click_key','count'),turn=('is_rewarded','sum')).reindex(weekday_order).reset_index()
    wd_norm['cvr'] = (wd_norm['turn']/wd_norm['clk']*100).round(2)
    print("[E. 요일별 패턴]")
    print(f"  {'요일':>4} {'58클릭':>8} {'58CVR%':>8} {'정상클릭':>8} {'정상CVR%':>9}")
    for i, row in wd_58.iterrows():
        norm_row = wd_norm[wd_norm['weekday']==row['weekday']]
        n_cvr = norm_row['cvr'].values[0] if len(norm_row)>0 else 0
        n_clk = norm_row['clk'].values[0] if len(norm_row)>0 else 0
        print(f"  {row['kor']:>4} {row['clk']:>8,.0f} {row['cvr']:>8.2f} {n_clk:>8,.0f} {n_cvr:>9.2f}")
else:
    print("[E. 요일별 패턴 — report_master 기준]")
    wd_rpt = (
        report_master[report_master['mda_idx']==MDA_TARGET]
        .groupby('weekday').agg(clk=('rpt_time_clk','sum'),turn=('rpt_time_turn','sum')).reindex(weekday_order).reset_index()
    )
    wd_rpt['cvr'] = (wd_rpt['turn']/wd_rpt['clk']*100).round(2)
    wd_rpt['kor'] = wd_rpt['weekday'].map(weekday_kor_map)
    print(wd_rpt[['kor','clk','turn','cvr']].to_string(index=False))

[E. 요일별 패턴 — report_master 기준]
kor     clk   turn   cvr
  월  632499 231433 36.59
  화  839050 336291 40.08
  수  946719 401037 42.36
  목 1303929 605830 46.46
  금 1378521 671038 48.68
  토  982178 413858 42.14
  일  709594 266005 37.49


In [31]:
# F. 광고유형별 분포
loss_by_type = (
    mda58[(mda58['is_rewarded']==1)&(mda58['row_label']=='극단값')]
    .groupby('ads_type_name')['show_cost'].sum()
    .reset_index().rename(columns={'show_cost':'loss_show'})
)
type_58 = (
    mda58.groupby('ads_type_name')
    .agg(total_clk=('click_key','count'), total_turn=('is_rewarded','sum'),
         extreme_cnt=('row_label', lambda x: (x=='극단값').sum()))
    .reset_index()
)
type_58['cvr']         = (type_58['total_turn']/type_58['total_clk']*100).round(2)
type_58['extreme_pct'] = (type_58['extreme_cnt']/type_58['total_clk']*100).round(1)
type_58 = type_58.merge(loss_by_type, on='ads_type_name', how='left').fillna(0)
type_58['loss_pct'] = (type_58['loss_show']/mda58_loss*100).round(1) if mda58_loss>0 else 0

print("[F. 광고유형별 어뷰징 분포]")
print(f"  {'광고유형':12} {'클릭':>8} {'전환':>7} {'CVR%':>7} {'극단값%':>8} {'손실액':>12} {'손실비%':>8}")
for _, r in type_58.sort_values('loss_show', ascending=False).iterrows():
    print(f"  {str(r['ads_type_name']):12} {r['total_clk']:>8,.0f} {r['total_turn']:>7,.0f} {r['cvr']:>7.2f} {r['extreme_pct']:>8.1f} {r['loss_show']:>12,.0f} {r['loss_pct']:>8.1f}")

# G. IP 집중도 (클릭 팜 증거)
ip_58_detail = (
    mda58.groupby('user_ip')
    .agg(clk_cnt=('click_key','count'), turn_cnt=('is_rewarded','sum'),
         dvc_cnt=('dvc_idx','nunique'), extreme_cnt=('row_label', lambda x:(x=='극단값').sum()))
    .reset_index().sort_values('clk_cnt', ascending=False)
)
ip_58_detail['extreme_pct'] = (ip_58_detail['extreme_cnt']/ip_58_detail['clk_cnt']*100).round(1)
ip_58_detail['cvr_ip']      = (ip_58_detail['turn_cnt']/ip_58_detail['clk_cnt']*100).round(2)

print(f"\n[G. IP 집중도]  고유 IP: {len(ip_58_detail):,}개")
for n in [1,5,10,20,50]:
    top_clk = ip_58_detail.head(n)['clk_cnt'].sum()
    print(f"  상위 {n:>3}개 IP → {top_clk:>8,}클릭  ({top_clk/total_clk*100:.1f}%)")

print(f"\n  {'IP':20} {'클릭':>8} {'전환':>7} {'기기':>6} {'CVR%':>7} {'극단값%':>8}")
for _, r in ip_58_detail.head(10).iterrows():
    ip_masked = str(r['user_ip'])[:8] + '***'
    print(f"  {ip_masked:20} {r['clk_cnt']:>8,} {r['turn_cnt']:>7,} {r['dvc_cnt']:>6,} {r['cvr_ip']:>7.2f} {r['extreme_pct']:>8.1f}")

[F. 광고유형별 어뷰징 분포]
  광고유형               클릭      전환    CVR%     극단값%          손실액     손실비%
  설치형           229,889 112,031   48.73     96.9   19,589,400     60.2
  실행형            70,754  42,637   60.26     91.7   10,995,040     33.8
  참여형           182,151   1,159    0.64     67.6    1,942,892      6.0

[G. IP 집중도]  고유 IP: 5,359개
  상위   1개 IP →   12,315클릭  (2.6%)
  상위   5개 IP →   57,587클릭  (11.9%)
  상위  10개 IP →   88,131클릭  (18.3%)
  상위  20개 IP →  124,286클릭  (25.7%)
  상위  50개 IP →  187,682클릭  (38.9%)

  IP                         클릭      전환     기기    CVR%     극단값%
  18.183.4***            12,315   4,960 10,676   40.28    100.0
  54.168.2***            12,058   4,873 10,405   40.41    100.0
  13.158.7***            11,686   4,745 10,148   40.60    100.0
  54.168.1***            11,532   4,669 10,056   40.49    100.0
  54.248.2***             9,996   4,559  8,898   45.61    100.0
  52.195.2***             7,517   3,377  6,357   44.92    100.0
  52.194.1***             6,104   2,519  5,405 

In [32]:
# H. 기기 블랙리스트 교차
top_devices_all = set(device_stats.head(100)['dvc_idx'])
mda58_devices   = set(mda58[mda58['row_label']=='극단값']['dvc_idx'].unique())
overlap         = mda58_devices & top_devices_all
overlap_clk     = mda58[mda58['dvc_idx'].isin(overlap)]['click_key'].count()

print(f"[H. 기기 블랙리스트 교차]")
print(f"  mda 58 극단값 기기: {len(mda58_devices):,}개  |  Top100 교집합: {len(overlap)}개  |  교집합 클릭: {overlap_clk:,}건 ({overlap_clk/total_clk*100:.1f}%)")

# I. mda 58 제거 전후 CVR
all_clk    = len(df_integrated)
all_turn   = df_integrated['is_rewarded'].sum()
cvr_before = all_turn / all_clk * 100
excl58     = df_integrated[df_integrated['mda_idx'] != MDA_TARGET]
cvr_after  = excl58['is_rewarded'].sum() / len(excl58) * 100
excl58_ext = df_integrated[~((df_integrated['mda_idx']==MDA_TARGET) & (df_integrated['row_label']=='극단값'))]
cvr_ext    = excl58_ext['is_rewarded'].sum() / len(excl58_ext) * 100

print(f"\n[I. mda 58 제거 전후 CVR]")
print(f"  제거 전: {cvr_before:.2f}%  →  mda 58 제거 후: {cvr_after:.2f}%  ({cvr_after-cvr_before:+.2f}%p)")
print(f"  극단값만 제거 시: {cvr_ext:.2f}%  ({cvr_ext-cvr_before:+.2f}%p)")

# 핵심 수치 요약
print(f"\n[핵심 요약 — mda {MDA_TARGET}]")
print(f"  클릭 {total_clk:,}  |  극단값 {label_pct.get('극단값',0):.1f}%  어뷰징 {label_pct.get('어뷰징',0):.1f}%  비정상 {label_pct.get('극단값',0)+label_pct.get('어뷰징',0):.1f}%")
print(f"  CVR {cvr_mda58:.2f}% (정상 {cvr_normal:.2f}%, 격차 {cvr_mda58-cvr_normal:+.2f}%p)")
print(f"  확정 손실 {mda58_loss:,.0f}원  ({mda58_loss/total_confirmed_loss*100:.1f}%)")
print(f"  제거 시 CVR 회복: {cvr_after-cvr_before:+.2f}%p  |  상위 10개 IP 클릭: {ip_58_detail.head(10)['clk_cnt'].sum():,}건 ({ip_58_detail.head(10)['clk_cnt'].sum()/total_clk*100:.1f}%)")

[H. 기기 블랙리스트 교차]
  mda 58 극단값 기기: 153,593개  |  Top100 교집합: 9개  |  교집합 클릭: 2,420건 (0.5%)

[I. mda 58 제거 전후 CVR]
  제거 전: 8.74%  →  mda 58 제거 후: 8.04%  (-0.70%p)
  극단값만 제거 시: 8.05%  (-0.69%p)

[핵심 요약 — mda 58]
  클릭 482,794  |  극단값 85.1%  어뷰징 14.4%  비정상 99.5%
  CVR 32.28% (정상 44.74%, 격차 -12.46%p)
  확정 손실 32,527,332원  (48.5%)
  제거 시 CVR 회복: -0.70%p  |  상위 10개 IP 클릭: 88,131건 (18.3%)


## 비용 손실 검증 분석

In [33]:
target_ym = report_master['year_month'].max()

rewarded_rows       = df_integrated[df_integrated['is_rewarded']==1]
total_show_rewarded = rewarded_rows['show_cost'].sum()
total_turn_v        = len(rewarded_rows)
total_earn_rewarded = rewarded_rows['earn_cost'].sum()
cpa_show = total_show_rewarded / total_turn_v if total_turn_v > 0 else 0
cpa_earn = total_earn_rewarded / total_turn_v if total_turn_v > 0 else 0

print(f"[V-1] 과금 구조  전환: {total_turn_v:,}건  show CPA: {cpa_show:.1f}원  earn CPA: {cpa_earn:.1f}원")

# 손실 3단계 분해
mask_confirmed_v  = (df_integrated['is_rewarded']==1) & (df_integrated['row_label']=='극단값')
mask_manageable_v = (df_integrated['is_rewarded']==1) & (df_integrated['row_label']=='어뷰징')
mask_normal_rwd   = (df_integrated['is_rewarded']==1) & (df_integrated['row_label']=='정상')

confirmed_show  = df_integrated.loc[mask_confirmed_v,  'show_cost'].sum()
manageable_show = df_integrated.loc[mask_manageable_v, 'show_cost'].sum()
normal_show     = df_integrated.loc[mask_normal_rwd,   'show_cost'].sum()
confirmed_earn  = df_integrated.loc[mask_confirmed_v,  'earn_cost'].sum()
manageable_earn = df_integrated.loc[mask_manageable_v, 'earn_cost'].sum()
total_loss_show = confirmed_show + manageable_show
total_loss_earn = confirmed_earn + manageable_earn

print(f"\n[V-2] 손실 3단계 분해")
print(f"  {'구분':10}  {'전환':>10}  {'show_cost':>14}  {'earn_cost':>14}  {'show비중':>8}")
for name, mask in [('확정 손실',mask_confirmed_v),('관리 손실',mask_manageable_v),('정상 비용',mask_normal_rwd)]:
    turn = int(mask.sum()); show = df_integrated.loc[mask,'show_cost'].sum(); earn = df_integrated.loc[mask,'earn_cost'].sum()
    print(f"  {name:10}  {turn:>10,}건  {show:>14,.0f}원  {earn:>14,.0f}원  {show/total_show_rewarded*100:>7.1f}%")
print(f"  총 손실 show: {total_loss_show:,.0f}원  ({total_loss_show/1e8:.2f}억)  |  earn: {total_loss_earn:,.0f}원  ({total_loss_earn/1e8:.2f}억)")

# 기회 손실
normal_mda_df   = df_integrated[df_integrated['Risk_Label']=='정상']
cvr_normal_mda  = normal_mda_df['is_rewarded'].sum()/len(normal_mda_df)*100 if len(normal_mda_df)>0 else 0
abnormal_mask   = df_integrated['row_label'].isin(['극단값','어뷰징'])
abnormal_clk    = abnormal_mask.sum()
actual_abn_turn = int(df_integrated[abnormal_mask & (df_integrated['is_rewarded']==1)]['is_rewarded'].sum())
expected_turn   = abnormal_clk * (cvr_normal_mda / 100)
opp_turn_loss   = expected_turn - actual_abn_turn
opp_cost_loss   = opp_turn_loss * cpa_show

print(f"\n[V-3] 기회 손실  정상CVR: {cvr_normal_mda:.2f}%  비정상클릭: {abnormal_clk:,}건")
print(f"  기회 손실 전환: {opp_turn_loss:,.0f}건  |  기회 손실 금액: {opp_cost_loss:,.0f}원  ({opp_cost_loss/1e8:.2f}억)")

# report_master 대조
one_m    = report_master[report_master['year_month']==target_ym]
rm_scost = one_m['rpt_time_scost'].sum(); rm_earn = one_m['rpt_time_earn'].sum()
rm_clk   = one_m['rpt_time_clk'].sum();  rm_turn = one_m['rpt_time_turn'].sum()
print(f"\n[report_master 대조 — {target_ym}]  scost: {rm_scost:,.0f}원  earn: {rm_earn:,.0f}원  역마진: {rm_earn-rm_scost:,.0f}원")

# 매체별 확정 손실 Top10
mda_loss = (
    df_integrated[mask_confirmed_v].groupby('mda_idx')
    .agg(turn_cnt=('is_rewarded','sum'), show_loss=('show_cost','sum'), earn_loss=('earn_cost','sum'))
    .reset_index().sort_values('show_loss', ascending=False).reset_index(drop=True)
)
mda_loss['loss_pct'] = (mda_loss['show_loss']/confirmed_show*100).round(1)
mda_loss['cum_pct']  = mda_loss['loss_pct'].cumsum().round(1)

print(f"\n[V-4] 매체별 확정 손실 Top 10")
for rank in range(min(10, len(mda_loss))):
    r = mda_loss.iloc[rank]
    print(f"  {rank+1:2}위  mda {int(r['mda_idx']):>4}  전환 {int(r['turn_cnt']):>8,}건  show {r['show_loss']:>14,.0f}원  {r['loss_pct']:>5.1f}%  누적 {r['cum_pct']:>5.1f}%")
print(f"  상위 2개: {mda_loss.head(2)['loss_pct'].sum():.1f}%  |  상위 3개: {mda_loss.head(3)['loss_pct'].sum():.1f}%")

# row_label별 전체 요약
print(f"\n[V-6] row_label별 전체 요약")
for lbl in ['극단값','어뷰징','미적립','정상']:
    sub = df_integrated[df_integrated['row_label']==lbl]
    if len(sub)==0: continue
    clk_cnt = len(sub); turn_cnt = int(sub['is_rewarded'].sum()); show_sum = sub['show_cost'].sum()
    cvr_v   = turn_cnt/clk_cnt*100 if clk_cnt>0 else 0
    print(f"  {lbl:6}  클릭 {clk_cnt:>12,} ({clk_cnt/len(df_integrated)*100:>5.1f}%)  전환 {turn_cnt:>9,}  CVR {cvr_v:>6.2f}%  show {show_sum:>14,.0f}  ({show_sum/total_show_rewarded*100:>5.1f}%)")

[V-1] 과금 구조  전환: 1,470,533건  show CPA: 392.5원  earn CPA: 158.4원

[V-2] 손실 3단계 분해
  구분                  전환       show_cost       earn_cost    show비중
  확정 손실          530,791건      67,083,271원      55,246,949원     11.6%
  관리 손실          128,177건     193,575,035원      27,365,703원     33.5%
  정상 비용          811,565건     316,541,152원     150,257,463원     54.8%
  총 손실 show: 260,658,306원  (2.61억)  |  earn: 82,612,652원  (0.83억)

[V-3] 기회 손실  정상CVR: 44.74%  비정상클릭: 14,170,417건
  기회 손실 전환: 5,680,936건  |  기회 손실 금액: 2,229,826,228원  (22.30억)

[report_master 대조 — 2025-08]  scost: 469,105,219원  earn: 254,895,411원  역마진: -214,209,808원

[V-4] 매체별 확정 손실 Top 10
   1위  mda   58  전환  148,809건  show     32,527,332원   48.5%  누적  48.5%
   2위  mda  563  전환  310,629건  show     18,149,774원   27.1%  누적  75.6%
   3위  mda  342  전환   11,358건  show      3,170,140원    4.7%  누적  80.3%
   4위  mda  761  전환   13,419건  show      2,691,187원    4.0%  누적  84.3%
   5위  mda  343  전환    5,670건  show      1,587,600원    2.4%  누적  86

## CVR 격차 분석 — 위험 등급별 전환율 비교

In [34]:
risk_order = ['정상','경고','위험','매우위험','미분류']

risk_summary = (
    df_integrated.groupby('Risk_Label')
    .agg(매체수=('mda_idx','nunique'), 클릭수=('click_key','count'),
         전환수=('is_rewarded','sum'), show합산=('show_cost','sum'), earn합산=('earn_cost','sum'))
    .reset_index()
)
risk_summary['CVR%']     = (risk_summary['전환수']/risk_summary['클릭수']*100).round(2)
risk_summary['클릭비중'] = (risk_summary['클릭수']/risk_summary['클릭수'].sum()*100).round(1)
risk_summary['비용비중'] = (risk_summary['show합산']/risk_summary['show합산'].sum()*100).round(1)
risk_summary['전환비중'] = (risk_summary['전환수']/risk_summary['전환수'].sum()*100).round(1)

cvr_normal_v = float(risk_summary.loc[risk_summary['Risk_Label']=='정상','CVR%'].values[0])
risk_summary['정상대비차'] = (risk_summary['CVR%'] - cvr_normal_v).round(2)

print(f"[C-1] 위험 등급별 CVR  (정상 기준: {cvr_normal_v:.2f}%)")
print(f"  {'등급':8}  {'매체수':>6}  {'클릭수':>10}  {'클릭%':>6}  {'전환수':>8}  {'CVR%':>7}  {'정상대비':>9}  {'비용%':>6}  {'전환%':>6}")
for lbl in risk_order:
    row = risk_summary[risk_summary['Risk_Label']==lbl]
    if len(row)==0: continue
    r = row.iloc[0]
    print(f"  {r['Risk_Label']:8}  {int(r['매체수']):>6}개  {int(r['클릭수']):>10,}  {r['클릭비중']:>5.1f}%  {int(r['전환수']):>8,}  {r['CVR%']:>6.2f}%  {r['정상대비차']:>+8.2f}%p  {r['비용비중']:>5.1f}%  {r['전환비중']:>5.1f}%")

# 등급별 클릭 품질
print(f"\n[C-2] 위험 등급별 클릭 품질 구성")
for lbl in risk_order:
    sub = df_integrated[df_integrated['Risk_Label']==lbl]
    if len(sub)==0: continue
    total = len(sub)
    ext = (sub['row_label']=='극단값').sum()/total*100; abu = (sub['row_label']=='어뷰징').sum()/total*100
    unr = (sub['row_label']=='미적립').sum()/total*100; nor = (sub['row_label']=='정상').sum()/total*100
    print(f"  {lbl:8}  총클릭 {total:>10,}  극단값 {ext:>5.1f}%  어뷰징 {abu:>5.1f}%  미적립 {unr:>5.1f}%  정상 {nor:>5.1f}%  비정상합 {ext+abu+unr:>5.1f}%")

# 정상 vs 매우위험 상세
cvr_vd = float(risk_summary.loc[risk_summary['Risk_Label']=='매우위험','CVR%'].values[0])
print(f"\n[C-4] 정상 vs 매우위험 CVR 격차: {cvr_normal_v-cvr_vd:.2f}%p")

# 비용 vs 전환 불균형
print(f"\n[C-5] 비용 기여 vs 전환 기여 불균형")
for lbl in risk_order:
    row = risk_summary[risk_summary['Risk_Label']==lbl]
    if len(row)==0: continue
    r = row.iloc[0]
    gap = r['비용비중']-r['전환비중']; cpc = r['show합산']/r['클릭수'] if r['클릭수']>0 else 0
    marker = ' ◀ 비용 낭비' if gap>5 else (' ◀ 이익 기여' if gap<-5 else '')
    print(f"  {r['Risk_Label']:8}  비용 {r['비용비중']:>5.1f}%  전환 {r['전환비중']:>5.1f}%  불균형 {gap:>+6.1f}%p  CPC {cpc:>7.1f}원{marker}")

# 월별 CVR 추이
mda_risk_map = df_integrated[['mda_idx','Risk_Label']].drop_duplicates('mda_idx')
rm_with_risk = report_master.merge(mda_risk_map, on='mda_idx', how='left')
monthly_cvr  = (
    rm_with_risk.groupby(['year_month','Risk_Label'])
    .agg(clk=('rpt_time_clk','sum'), turn=('rpt_time_turn','sum')).reset_index()
)
monthly_cvr['CVR%'] = (monthly_cvr['turn']/monthly_cvr['clk']*100).round(2)
months_all = sorted(monthly_cvr['year_month'].unique())

print(f"\n[C-3] 월별 위험 등급별 CVR 추이 (최근 6개월)")
print(f"  {'등급':8}", end='')
for m in months_all[-6:]:
    print(f"  {str(m)[2:]:>8}", end='')
print()
for lbl in risk_order:
    print(f"  {lbl:8}", end='')
    for m in months_all[-6:]:
        val = monthly_cvr[(monthly_cvr['year_month']==m)&(monthly_cvr['Risk_Label']==lbl)]['CVR%']
        print(f"  {f'{val.values[0]:.2f}%' if len(val)>0 else '   N/A':>8}", end='')
    print()

[C-1] 위험 등급별 CVR  (정상 기준: 44.74%)
  등급           매체수         클릭수     클릭%       전환수     CVR%       정상대비     비용%     전환%
  정상           135개     743,310    4.4%   332,560   44.74%     +0.00%p   12.6%   22.6%
  경고            35개   1,384,490    8.2%   601,253   43.43%     -1.31%p   75.4%   40.9%
  위험            12개     109,944    0.7%    31,325   28.49%    -16.25%p    1.2%    2.1%
  매우위험           7개  14,593,310   86.7%   505,395    3.46%    -41.28%p   10.8%   34.4%

[C-2] 위험 등급별 클릭 품질 구성
  정상        총클릭    743,310  극단값   0.3%  어뷰징   3.0%  미적립  53.4%  정상  43.4%  비정상합  56.6%
  경고        총클릭  1,384,490  극단값   2.7%  어뷰징  17.5%  미적립  45.5%  정상  34.4%  비정상합  65.6%
  위험        총클릭    109,944  극단값  22.3%  어뷰징  44.7%  미적립  26.6%  정상   6.3%  비정상합  93.7%
  매우위험      총클릭 14,593,310  극단값  43.3%  어뷰징  51.2%  미적립   5.4%  정상   0.0%  비정상합 100.0%

[C-4] 정상 vs 매우위험 CVR 격차: 41.28%p

[C-5] 비용 기여 vs 전환 기여 불균형
  정상        비용  12.6%  전환  22.6%  불균형  -10.0%p  CPC    97.6원 ◀ 이익 기여
  경고        비용  75.4%  전환  40.9% 